In [1]:
import os
from dotenv import load_dotenv
load_dotenv(".env", override=True)

LLM_MODEL = os.getenv("LLM_MODEL", "gemini/gemini-flash-latest") 
from memory_store import get_memory_namespace
MEMORY_NAMESPACE = get_memory_namespace()

print("Google key      :", bool(os.getenv("GEMINI_API_KEY")), "(required for the LLM)")
print("Supermemory key :", bool(os.getenv("SUPERMEMORY_API_KEY")), "(required for the memory store)")
print("LLM model       :", LLM_MODEL)
print("Memory namespace:", MEMORY_NAMESPACE)

Google key      : True (required for the LLM)
Supermemory key : True (required for the memory store)
LLM model       : gemini/gemini-flash-latest
Memory namespace: loan_applicants


In [2]:
# Readiness check: confirms libraries import and keys are present. Calls no paid API.
status = {}
for name, module in [("litellm", "litellm"), ("supermemory", "supermemory"), ("tiktoken", "tiktoken")]:
    try:
        __import__(module); status[name] = "ok"
    except Exception as e:
        status[name] = f"import failed: {e}"
status["GEMINI_API_KEY"]      = "ok" if os.getenv("GEMINI_API_KEY") else "missing (LLM will fail)"
status["SUPERMEMORY_API_KEY"] = "ok" if os.getenv("SUPERMEMORY_API_KEY") else "missing (memory store will fail)"

print("Environment readiness")
print("---------------------")
for k, v in status.items():
    print(f"  {k:>20} : {v}")

Environment readiness
---------------------
               litellm : ok
           supermemory : ok
              tiktoken : ok
        GEMINI_API_KEY : ok
   SUPERMEMORY_API_KEY : ok


## Write the long-term memories

In [3]:
import supermemory
from memory_store import recall_memories, remember_memory
mem = supermemory.Supermemory(api_key=os.getenv("SUPERMEMORY_API_KEY"))

# Write a memory into the store (each is stored as a document, tagged with its applicant ID).
def remember(text, applicant_id, **extra):
    return remember_memory(mem, MEMORY_NAMESPACE, text, applicant_id, **extra)

def recall(query, app_id, k=1):
    return recall_memories(mem, MEMORY_NAMESPACE, query, app_id, k)

In [4]:
# load a json file which has a list of dictionaries, each representing a past decision
import json

with open("../data/wealthpilot/mock_api/past_decisions.json", "r") as f:
    past_decisions_json = json.load(f)

In [9]:
past_decisions_json[0]

{'decision_id': 'ASH-D-00000',
 'applicant_id': 'ASH-A-00037',
 'decided_on': '2025-07-05',
 'outcome': 'declined',
 'internal_grade': 'A1',
 'approved_amount_inr': 1500000,
 'reason_code': 'ADVERSE_BUREAU',
 'human_signatory': 'ASH-U-002'}

In [7]:
# adding to the memory
for decision in past_decisions_json[0:100]:
    remember(str(decision), decision['applicant_id'], **{"Decision ID": decision['decision_id']})

In [10]:
# retrieval
for applicant_id, text, score in recall("Are there any past history of applicant ASH-A-00037?", "ASH-A-00037", k=1):
    print(text)

{'decision_id': 'ASH-D-00000', 'applicant_id': 'ASH-A-00037', 'decided_on': '2025-07-05', 'outcome': 'declined', 'internal_grade': 'A1', 'approved_amount_inr': 1500000, 'reason_code': 'ADVERSE_BUREAU', 'human_signatory': 'ASH-U-002'}


In [11]:
score

0.63490415

## Retrieval Evaluation

In [13]:
# Golden decisions measure retrieval relevance; applicant IDs measure isolation.
evaluation_cases = [
    {
        "query": f"past decision for applicant {decision['applicant_id']} {decision['reason_code']}",
        "applicant_id": decision["applicant_id"],
        "decision_id": decision["decision_id"],
    }
    for decision in past_decisions_json[:10]
]

evaluation_cases[:3]

[{'query': 'past decision for applicant ASH-A-00037 ADVERSE_BUREAU',
  'applicant_id': 'ASH-A-00037',
  'decision_id': 'ASH-D-00000'},
 {'query': 'past decision for applicant ASH-A-00168 APPROVED_STANDARD',
  'applicant_id': 'ASH-A-00168',
  'decision_id': 'ASH-D-00001'},
 {'query': 'past decision for applicant ASH-A-00225 THIN_FILE',
  'applicant_id': 'ASH-A-00225',
  'decision_id': 'ASH-D-00002'}]

In [14]:
retrieval_k = 3
ranks = []
isolation_violations = 0
for case in evaluation_cases:
    results = recall(case["query"], case["applicant_id"], k=retrieval_k)
    isolation_violations += sum(
        retrieved_applicant_id != case["applicant_id"]
        for retrieved_applicant_id, _, _ in results
    )
    rank = next(
        (
            index
            for index, (_, text, _) in enumerate(results, start=1)
            if case["decision_id"] in text
        ),
        None,
    )
    ranks.append(rank)

hit_at_3 = sum(rank is not None for rank in ranks) / len(ranks)
mrr_at_3 = sum(1 / rank for rank in ranks if rank is not None) / len(ranks)
print(f"Hit@{retrieval_k}: {hit_at_3:.2f}")
print(f"MRR@{retrieval_k}:  {mrr_at_3:.2f}")

print(f"Applicant-isolation violations: {isolation_violations}")
assert hit_at_3 >= 0.8, "Memory retrieval quality is below the 0.80 Hit@3 target"
assert isolation_violations == 0, "A memory from another applicant was returned"

Hit@3: 1.00
MRR@3:  1.00
Applicant-isolation violations: 0
